# Phase 04B Replacement: ExpansionNet v2 → S1 Sequence-KD GPU Preflight

Notebook này **thay thế** Phase 04B Logit KD cũ của CLIP. Output cũ chỉ còn
vai trò lịch sử và không được nối vào pipeline chính.

Preflight dùng 64 ảnh Train+Restval và pseudo-caption đã khóa tại Phase 03E:

1. xác minh Phase 1, Phase 2A và Phase 03E;
2. ghép human caption với ExpansionNet pseudo-caption theo `image_id`;
3. strict-load S1 MobileNetV3-Large và shared decoder seed 42;
4. tính `0.5 × CE_human + 0.5 × CE_sequence`;
5. backward và kiểm tra toàn bộ gradient Student;
6. thực hiện một optimizer step bị loại bỏ;
7. audit checkpoint/restore nguyên tử và khôi phục initialization;
8. cấp quyền cho Phase 04C replacement training readiness.

## Kaggle inputs bắt buộc

1. Phase 01 Formal Output.
2. Phase 02A Shared Student Pipeline.
3. Phase 03E ExpansionNet v2 Replacement Handoff.
4. MS COCO 2014 images.

Không cần gắn CLIP, ExpansionNet checkpoint hoặc output Phase 04B cũ.

## Runtime

- GPU T4 hoặc mạnh hơn.
- Internet Off.
- Chạy tuần tự Cell 1 → Cell 6.
- Không đọc Held-out Test Parquet hoặc Test images.

## Quan hệ với đề cương

Cell này kiểm chứng cơ chế chưng cất chuỗi, gradient, checkpoint và khả năng
tái lập trước full training; phù hợp phần phương pháp, cài đặt và thực nghiệm
KD trong khóa luận.


## Cell 1 — Environment, authorization và immutable inputs

**Công việc**

- xác minh authorization từ Phase 03E;
- xác minh 13/13 artifact handoff và 38/38 artifact Phase 2A;
- khóa S1 initialization, corpus Parquet và loss contract;
- bật strict deterministic GPU mode.

**Đầu ra dự kiến:** `Status: PASSED`, tiến độ `1 / 6`.


In [1]:
import copy
import csv
import hashlib
import json
import math
import os
import platform
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch


STARTED = time.perf_counter()
SEED = 42
INPUT_ROOT = Path(
    os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input")
)
OUTPUT_ROOT = Path(
    os.environ.get(
        "KLCN_OUTPUT_ROOT",
        "/kaggle/working//"
        "phase4_teacher_a_s1_sequence_kd_preflight",
    )
)
AUDIT_DIR = OUTPUT_ROOT / "audits"
CONFIG_DIR = OUTPUT_ROOT / "configs"
DATA_DIR = OUTPUT_ROOT / "data"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [
    OUTPUT_ROOT, AUDIT_DIR, CONFIG_DIR, DATA_DIR, MANIFEST_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def exactly_one(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name}; tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, target)
    return target


def set_deterministic(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=False)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)


if not torch.cuda.is_available():
    raise RuntimeError(
        "Phase 04B replacement yêu cầu GPU. "
        "Chọn Kaggle GPU T4 rồi chạy lại."
    )
set_deterministic(SEED)
device = torch.device("cuda")

PHASE1_READY_PATH = exactly_one(
    "PHASE1_FORMAL_DELIVERABLES_READY.json"
)
PHASE2_READY_PATH = exactly_one(
    "PHASE2_SHARED_TRAINING_BUNDLE_READY.json"
)
PHASE3E_COMPLETE_PATH = exactly_one(
    "PHASE3_TEACHER_A_EXPANSIONNET_V2_HANDOFF_COMPLETE.json"
)
AUTH_PATH = exactly_one(
    "PHASE4_TEACHER_A_S1_SEQUENCE_KD_PREFLIGHT_AUTHORIZED.json"
)

phase1_ready = read_json(PHASE1_READY_PATH)
phase2_ready = read_json(PHASE2_READY_PATH)
phase3e_complete = read_json(PHASE3E_COMPLETE_PATH)
authorization = read_json(AUTH_PATH)
assert phase2_ready["status"] == "ready"
assert phase2_ready["full_training_authorized"] is True
assert phase3e_complete["status"] == "complete"
assert phase3e_complete["replacement_decision"] == "locked"
assert phase3e_complete["selected_method"] == "sequence_kd"
assert phase3e_complete["direct_logit_kd_allowed"] is False
assert phase3e_complete["phase4_preflight_authorized"] is True
assert authorization["status"] == "authorized"
assert authorization["experiment_id"] == "P4A_EXPANSIONNET_S1_SEQKD"
assert authorization["teacher"] == "ExpansionNet v2"
assert authorization["student"] == "S1 MobileNetV3-Large"
assert authorization["selected_method"] == "sequence_kd"
for marker in [phase3e_complete, authorization]:
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False

PHASE2_ROOT = PHASE2_READY_PATH.parent
PHASE3E_ROOT = PHASE3E_COMPLETE_PATH.parent


def verify_manifest(root, csv_path, expected_rows):
    with Path(csv_path).open(
        "r", encoding="utf-8", newline=""
    ) as handle:
        rows = list(csv.DictReader(handle))
    assert len(rows) == expected_rows
    for row in rows:
        artifact = resolve_inside(root, row["relative_path"])
        assert artifact.is_file(), artifact
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256_file(artifact) == row["sha256"]
    return rows


phase3e_manifest_ref = phase3e_complete["manifest"]["csv"]
PHASE3E_MANIFEST = resolve_inside(
    PHASE3E_ROOT, phase3e_manifest_ref["relative_path"]
)
assert sha256_file(PHASE3E_MANIFEST) == phase3e_manifest_ref["sha256"]
phase3e_rows = verify_manifest(PHASE3E_ROOT, PHASE3E_MANIFEST, 13)

phase2_manifest_ref = phase2_ready["manifests"]["csv"]
PHASE2_MANIFEST = resolve_inside(
    PHASE2_ROOT, phase2_manifest_ref["relative_path"]
)
assert sha256_file(PHASE2_MANIFEST) == phase2_manifest_ref["sha256"]
phase2_rows = verify_manifest(PHASE2_ROOT, PHASE2_MANIFEST, 38)

handoff_ref = authorization["handoff_contract"]
HANDOFF_PATH = resolve_inside(
    PHASE3E_ROOT, handoff_ref["relative_path"]
)
assert sha256_file(HANDOFF_PATH) == handoff_ref["sha256"]
HANDOFF = read_json(HANDOFF_PATH)
assert HANDOFF["method"] == "sequence_kd"
assert HANDOFF["loss"]["human_caption_ce_weight"] == 0.5
assert HANDOFF["loss"]["teacher_sequence_ce_weight"] == 0.5
assert HANDOFF["direct_logit_kd_allowed"] is False
assert HANDOFF["feature_kd_in_phase4_allowed"] is False
assert HANDOFF["test_set_status"] == "sealed"

parquet_ref = HANDOFF["pseudo_caption_parquet"]
PSEUDO_PARQUET = resolve_inside(
    PHASE3E_ROOT, parquet_ref["relative_path"]
)
assert PSEUDO_PARQUET.stat().st_size == parquet_ref["size_bytes"]
assert sha256_file(PSEUDO_PARQUET) == parquet_ref["sha256"]

S1_INITIAL_CHECKPOINT = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["required_artifacts"]["s1_encoder_checkpoint"],
)
DECODER_INITIAL_CHECKPOINT = resolve_inside(
    PHASE2_ROOT,
    phase2_ready["required_artifacts"]["shared_decoder_checkpoint"],
)
DECODER_CONTRACT = resolve_inside(
    PHASE2_ROOT,
    "configs/decoder/shared_transformer_decoder_contract.json",
)

write_json(
    AUDIT_DIR / "phase4b_cell01_environment_inputs.json",
    {
        "audit_version": "phase4b_expansionnet_cell01_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        "seed": SEED,
        "strict_deterministic": True,
        "phase2_manifest_rows": len(phase2_rows),
        "phase3e_manifest_rows": len(phase3e_rows),
        "handoff_contract_sha256": sha256_file(HANDOFF_PATH),
        "pseudo_parquet_sha256": sha256_file(PSEUDO_PARQUET),
        "direct_logit_kd_allowed": False,
        "model_loaded": False,
        "dataset_read": False,
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - STARTED,
    },
)
print("=" * 72)
print("PHASE 04B EXPANSIONNET -> S1 CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Experiment                : P4A_EXPANSIONNET_S1_SEQKD")
print("Development seed          : 42")
print("GPU                       :", torch.cuda.get_device_name(0))
print("Strict deterministic      : True")
print("Phase 2A manifest         : 38 / 38 VERIFIED")
print("Phase 03E manifest        : 13 / 13 VERIFIED")
print("Handoff / Parquet         : VERIFIED / VERIFIED")
print("Direct Logit KD           : PROHIBITED")
print("Model/dataset read        : False / False")
print("Test Set status           : SEALED")
print("Progress                  : 1 / 6 code cells")
print("Next                      : build fixed human/pseudo batch")


PHASE 04B EXPANSIONNET -> S1 CELL 1 STATUS
Status                    : PASSED
Experiment                : P4A_EXPANSIONNET_S1_SEQKD
Development seed          : 42
GPU                       : Tesla T4
Strict deterministic      : True
Phase 2A manifest         : 38 / 38 VERIFIED
Phase 03E manifest        : 13 / 13 VERIFIED
Handoff / Parquet         : VERIFIED / VERIFIED
Direct Logit KD           : PROHIBITED
Model/dataset read        : False / False
Test Set status           : SEALED
Progress                  : 1 / 6 code cells
Next                      : build fixed human/pseudo batch


## Cell 2 — Fixed 64-image human/pseudo supervision

**Công việc**

- lấy 64 ảnh đầu tiên theo corpus đã khóa;
- ghép đúng một human caption từ Phase 1 theo `image_id`;
- lấy trực tiếp S1 token IDs của ExpansionNet pseudo-caption;
- dựng đường dẫn ảnh Train+Restval, không mở Test.

**Đầu ra dự kiến:** 64/64 cặp supervision hợp lệ, tiến độ `2 / 6`.


In [2]:
STARTED = time.perf_counter()
import pyarrow.parquet as pq
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode


TRAIN_PARQUET = exactly_one("mscoco_train.parquet")
IMAGE_MANIFEST = exactly_one("mscoco_image_manifest.parquet")


def find_image_leaf(directory_name, prefix):
    candidates = []
    for path in INPUT_ROOT.rglob(directory_name):
        if path.is_dir() and next(path.glob(f"{prefix}*.jpg"), None):
            candidates.append(path.resolve())
    candidates = sorted(set(candidates))
    assert len(candidates) == 1, candidates
    return candidates[0]


TRAIN_LEAF = find_image_leaf("train2014", "COCO_train2014_")
VAL_LEAF = find_image_leaf("val2014", "COCO_val2014_")

path_rows = pq.read_table(
    IMAGE_MANIFEST,
    columns=[
        "image_id",
        "experiment_split",
        "source_partition",
        "image_path",
    ],
    filters=[("experiment_split", "=", "train")],
).to_pylist()
TRAIN_PATHS = {}
for row in path_rows:
    leaf = (
        TRAIN_LEAF
        if row["source_partition"] == "train2014"
        else VAL_LEAF
    )
    TRAIN_PATHS[int(row["image_id"])] = (
        leaf / Path(row["image_path"]).name
    )
assert len(TRAIN_PATHS) == 113287

pseudo_rows = pq.read_table(
    PSEUDO_PARQUET,
    columns=["global_index", "image_id", "student_token_ids"],
).slice(0, 64).to_pylist()
assert len(pseudo_rows) == 64
assert [int(row["global_index"]) for row in pseudo_rows] == list(range(64))
fixed_ids = [int(row["image_id"]) for row in pseudo_rows]
assert len(set(fixed_ids)) == 64
assert fixed_ids == sorted(fixed_ids)

human_by_image = {}
for batch in pq.ParquetFile(TRAIN_PARQUET).iter_batches(
    batch_size=4096,
    columns=["annotation_id", "image_id", "token_ids"],
):
    for row in batch.to_pylist():
        image_id = int(row["image_id"])
        if image_id not in fixed_ids or image_id in human_by_image:
            continue
        ids = [int(value) for value in row["token_ids"]]
        assert ids[0] == 1 and ids[-1] == 2
        human_by_image[image_id] = {
            "annotation_id": int(row["annotation_id"]),
            "human_ids": ids,
        }
    if len(human_by_image) == 64:
        break
assert len(human_by_image) == 64

records = []
for row in pseudo_rows:
    image_id = int(row["image_id"])
    pseudo_ids = [int(value) for value in row["student_token_ids"]]
    assert pseudo_ids[0] == 1 and pseudo_ids[-1] == 2
    assert 0 not in pseudo_ids and 3 not in pseudo_ids
    image_path = TRAIN_PATHS[image_id]
    assert image_path.is_file()
    records.append(
        {
            "global_index": int(row["global_index"]),
            "image_id": image_id,
            "annotation_id": human_by_image[image_id]["annotation_id"],
            "human_ids": human_by_image[image_id]["human_ids"],
            "pseudo_ids": pseudo_ids,
            "image_path": image_path,
        }
    )

FIXED64_PATH = DATA_DIR / "phase4b_expansionnet_fixed64_pairs.json"
write_json(
    FIXED64_PATH,
    {
        "schema_version": "phase4b_expansionnet_fixed64_v1",
        "split": "Train+Restval",
        "rows": [
            {
                key: (
                    value.as_posix()
                    if isinstance(value, Path)
                    else value
                )
                for key, value in row.items()
            }
            for row in records
        ],
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

human_tokens = sum(len(row["human_ids"]) - 2 for row in records)
pseudo_tokens = sum(len(row["pseudo_ids"]) - 2 for row in records)
write_json(
    AUDIT_DIR / "phase4b_cell02_fixed_supervision.json",
    {
        "audit_version": "phase4b_expansionnet_cell02_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "images": len(records),
        "unique_images": len({row["image_id"] for row in records}),
        "human_lexical_tokens": human_tokens,
        "pseudo_lexical_tokens": pseudo_tokens,
        "pseudo_unknown_tokens": 0,
        "pseudo_bos_eos_valid": True,
        "caption_text_read": False,
        "test_parquet_opened": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - STARTED,
    },
)
print("=" * 72)
print("PHASE 04B EXPANSIONNET -> S1 CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Fixed Train images        : 64 / 64")
print("Unique images             : 64")
print("Human lexical tokens      :", human_tokens)
print("Pseudo lexical tokens     :", pseudo_tokens)
print("Pseudo UNK tokens         : 0")
print("Pseudo BOS/EOS valid      : True")
print("Caption text read         : False")
print("Test Parquet opened       : False")
print("Test images read          : False")
print("Test Set status           : SEALED")
print("Progress                  : 2 / 6 code cells")
print("Next                      : reconstruct and strict-load S1")


PHASE 04B EXPANSIONNET -> S1 CELL 2 STATUS
Status                    : PASSED
Fixed Train images        : 64 / 64
Unique images             : 64
Human lexical tokens      : 644
Pseudo lexical tokens     : 607
Pseudo UNK tokens         : 0
Pseudo BOS/EOS valid      : True
Caption text read         : False
Test Parquet opened       : False
Test images read          : False
Test Set status           : SEALED
Progress                  : 2 / 6 code cells
Next                      : reconstruct and strict-load S1


## Cell 3 — Reconstruct S1 và prepare dual targets

**Công việc**

- dựng MobileNetV3-Large adapter và shared Transformer decoder;
- strict-load hai checkpoint initial seed 42;
- tạo tensor ảnh ImageNet-normalized;
- pad riêng human target và pseudo target.

**Đầu ra dự kiến:** 8.990.352 tham số, strict-load 2/2, tiến độ `3 / 6`.


In [3]:
STARTED = time.perf_counter()
import torch.nn as nn
import torch.nn.functional as F
from torch.torch_version import TorchVersion
from torchvision.models import mobilenet_v3_large


decoder_config = read_json(DECODER_CONTRACT)["architecture"]
assert int(decoder_config["vocabulary_size"]) == 10080
assert int(decoder_config["pad_id"]) == 0
assert int(decoder_config["maximum_decoder_steps"]) == 31


class SharedCaptionDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.vocabulary_size = int(config["vocabulary_size"])
        self.pad_id = int(config["pad_id"])
        self.d_model = int(config["d_model"])
        self.maximum_decoder_steps = int(
            config["maximum_decoder_steps"]
        )
        self.token_embedding = nn.Embedding(
            self.vocabulary_size,
            self.d_model,
            padding_idx=self.pad_id,
        )
        self.position_embedding = nn.Embedding(
            self.maximum_decoder_steps, self.d_model
        )
        self.embedding_dropout = nn.Dropout(
            float(config["dropout"])
        )
        layer = nn.TransformerDecoderLayer(
            d_model=self.d_model,
            nhead=int(config["nhead"]),
            dim_feedforward=int(config["dim_feedforward"]),
            dropout=float(config["dropout"]),
            activation=config["activation"],
            batch_first=True,
            norm_first=True,
            bias=True,
        )
        self.transformer_decoder = nn.TransformerDecoder(
            layer, num_layers=int(config["num_layers"])
        )
        self.final_norm = nn.LayerNorm(self.d_model)
        self.output_bias = nn.Parameter(
            torch.zeros(self.vocabulary_size)
        )

    def forward(self, ids, memory, mask):
        length = ids.shape[1]
        positions = torch.arange(
            length, device=ids.device
        ).unsqueeze(0).expand(ids.shape[0], -1)
        target = (
            self.token_embedding(ids) * math.sqrt(self.d_model)
            + self.position_embedding(positions)
        )
        target = self.embedding_dropout(target)
        causal = torch.triu(
            torch.ones(
                length,
                length,
                dtype=torch.bool,
                device=ids.device,
            ),
            diagonal=1,
        )
        hidden = self.transformer_decoder(
            target,
            memory,
            tgt_mask=causal,
            tgt_key_padding_mask=~mask.bool(),
        )
        hidden = self.final_norm(hidden)
        return F.linear(
            hidden, self.token_embedding.weight, self.output_bias
        )


class S1VisualEncoderAdapter(nn.Module):
    def __init__(self):
        super().__init__()
        model = mobilenet_v3_large(weights=None)
        self.backbone = model.features
        self.projection = nn.Linear(960, 256)
        self.visual_position_embedding = nn.Parameter(
            torch.empty(1, 49, 256)
        )
        self.output_norm = nn.LayerNorm(256)

    def forward(self, images):
        feature = self.backbone(images)
        tokens = feature.flatten(2).transpose(1, 2).contiguous()
        assert tokens.shape[1:] == (49, 960)
        return self.output_norm(
            self.projection(tokens)
            + self.visual_position_embedding
        )


STUDENT_ENCODER = S1VisualEncoderAdapter()
STUDENT_DECODER = SharedCaptionDecoder(decoder_config)
with torch.serialization.safe_globals([TorchVersion]):
    encoder_payload = torch.load(
        S1_INITIAL_CHECKPOINT,
        map_location="cpu",
        weights_only=True,
    )
    decoder_payload = torch.load(
        DECODER_INITIAL_CHECKPOINT,
        map_location="cpu",
        weights_only=True,
    )
encoder_load = STUDENT_ENCODER.load_state_dict(
    encoder_payload["state_dict"], strict=True
)
decoder_load = STUDENT_DECODER.load_state_dict(
    decoder_payload["state_dict"], strict=True
)
assert not encoder_load.missing_keys
assert not encoder_load.unexpected_keys
assert not decoder_load.missing_keys
assert not decoder_load.unexpected_keys
del encoder_payload, decoder_payload

assert sum(p.numel() for p in STUDENT_ENCODER.parameters()) == 3_231_024
assert sum(p.numel() for p in STUDENT_DECODER.parameters()) == 5_759_328
assert sum(
    p.numel()
    for module in [STUDENT_ENCODER, STUDENT_DECODER]
    for p in module.parameters()
) == 8_990_352

INITIAL_ENCODER = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_ENCODER.state_dict().items()
}
INITIAL_DECODER = {
    key: value.detach().cpu().clone()
    for key, value in STUDENT_DECODER.state_dict().items()
}
STUDENT_ENCODER = STUDENT_ENCODER.to(device)
STUDENT_DECODER = STUDENT_DECODER.to(device)

student_transform = transforms.Compose(
    [
        transforms.Resize(
            256, interpolation=InterpolationMode.BICUBIC
        ),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
        ),
    ]
)


def pad_full(sequences):
    maximum = max(len(sequence) for sequence in sequences)
    result = torch.zeros(
        len(sequences), maximum, dtype=torch.long
    )
    for index, sequence in enumerate(sequences):
        result[index, :len(sequence)] = torch.tensor(sequence)
    return result


smoke_records = records[:8]
images = []
for row in smoke_records:
    with Image.open(row["image_path"]) as source:
        images.append(student_transform(source.convert("RGB")))
STUDENT_IMAGES = torch.stack(images).to(device)
HUMAN_FULL = pad_full(
    [row["human_ids"] for row in smoke_records]
).to(device)
PSEUDO_FULL = pad_full(
    [row["pseudo_ids"] for row in smoke_records]
).to(device)
assert HUMAN_FULL.shape[1] <= 32
assert PSEUDO_FULL.shape[1] <= 32


def targets(full):
    return full[:, :-1], full[:, 1:], full[:, :-1].ne(0)


HUMAN_INPUT, HUMAN_LABELS, HUMAN_MASK = targets(HUMAN_FULL)
PSEUDO_INPUT, PSEUDO_LABELS, PSEUDO_MASK = targets(PSEUDO_FULL)

write_json(
    AUDIT_DIR / "phase4b_cell03_student_model.json",
    {
        "audit_version": "phase4b_expansionnet_cell03_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "student_parameters": 8_990_352,
        "student_images": list(STUDENT_IMAGES.shape),
        "human_decoder_shape": list(HUMAN_INPUT.shape),
        "pseudo_decoder_shape": list(PSEUDO_INPUT.shape),
        "strict_checkpoint_loads": "2 / 2",
        "teacher_model_loaded": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - STARTED,
    },
)
print("=" * 72)
print("PHASE 04B EXPANSIONNET -> S1 CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("S1 parameters             : 8990352")
print("Strict checkpoint loads   : 2 / 2")
print("Images                    :", list(STUDENT_IMAGES.shape))
print("Human target              :", list(HUMAN_LABELS.shape))
print("Pseudo target             :", list(PSEUDO_LABELS.shape))
print("Teacher model loaded      : False")
print("Test Set status           : SEALED")
print("Progress                  : 3 / 6 code cells")
print("Next                      : combined loss and backward")


PHASE 04B EXPANSIONNET -> S1 CELL 3 STATUS
Status                    : PASSED
S1 parameters             : 8990352
Strict checkpoint loads   : 2 / 2
Images                    : [8, 3, 224, 224]
Human target              : [8, 13]
Pseudo target             : [8, 12]
Teacher model loaded      : False
Test Set status           : SEALED
Progress                  : 3 / 6 code cells
Next                      : combined loss and backward


## Cell 4 — Combined Sequence-KD loss và gradient

**Công việc**

- tính CE trên human target và pseudo target;
- kết hợp trọng số 0,5/0,5 theo handoff contract;
- kiểm tra gradient của toàn bộ Student hữu hạn và khác 0;
- xác nhận không dùng direct Teacher logits.

**Đầu ra dự kiến:** loss/gradient `PASSED`, tiến độ `4 / 6`.


In [4]:
STARTED = time.perf_counter()


def set_student_train():
    STUDENT_ENCODER.train()
    STUDENT_DECODER.train()
    frozen_batchnorm = 0
    for module in STUDENT_ENCODER.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()
            frozen_batchnorm += 1
    assert frozen_batchnorm == 46
    return frozen_batchnorm


def sequence_ce(logits, labels):
    return F.cross_entropy(
        logits.reshape(-1, logits.shape[-1]),
        labels.reshape(-1),
        ignore_index=0,
    )


set_student_train()
STUDENT_ENCODER.zero_grad(set_to_none=True)
STUDENT_DECODER.zero_grad(set_to_none=True)
memory = STUDENT_ENCODER(STUDENT_IMAGES)
human_logits = STUDENT_DECODER(HUMAN_INPUT, memory, HUMAN_MASK)
pseudo_logits = STUDENT_DECODER(PSEUDO_INPUT, memory, PSEUDO_MASK)
human_ce = sequence_ce(human_logits, HUMAN_LABELS)
pseudo_ce = sequence_ce(pseudo_logits, PSEUDO_LABELS)
combined_loss = 0.5 * human_ce + 0.5 * pseudo_ce
assert torch.isfinite(human_ce)
assert torch.isfinite(pseudo_ce)
assert torch.isfinite(combined_loss)
combined_loss.backward()

student_gradients = [
    parameter.grad
    for module in [STUDENT_ENCODER, STUDENT_DECODER]
    for parameter in module.parameters()
    if parameter.requires_grad
]
assert student_gradients
assert all(gradient is not None for gradient in student_gradients)
assert all(
    torch.isfinite(gradient).all()
    for gradient in student_gradients
)
gradient_norm = torch.sqrt(
    sum(
        gradient.detach().float().pow(2).sum()
        for gradient in student_gradients
    )
)
assert float(gradient_norm) > 0.0

write_json(
    AUDIT_DIR / "phase4b_cell04_loss_backward.json",
    {
        "audit_version": "phase4b_expansionnet_cell04_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "human_ce": float(human_ce.detach()),
        "teacher_sequence_ce": float(pseudo_ce.detach()),
        "combined_loss": float(combined_loss.detach()),
        "human_ce_weight": 0.5,
        "teacher_sequence_ce_weight": 0.5,
        "temperature": None,
        "gradient_l2": float(gradient_norm),
        "student_gradients_finite": True,
        "teacher_model_loaded": False,
        "direct_teacher_logits_used": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - STARTED,
    },
)
print("=" * 72)
print("PHASE 04B EXPANSIONNET -> S1 CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Human CE                  :", float(human_ce.detach()))
print("Teacher-sequence CE       :", float(pseudo_ce.detach()))
print("Combined loss             :", float(combined_loss.detach()))
print("Student gradient L2       :", float(gradient_norm))
print("Student gradients finite  : True")
print("Direct Teacher logits     : False")
print("Teacher model loaded      : False")
print("Test Set status           : SEALED")
print("Progress                  : 4 / 6 code cells")
print("Next                      : discarded optimizer/checkpoint pilot")


PHASE 04B EXPANSIONNET -> S1 CELL 4 STATUS
Status                    : PASSED
Human CE                  : 10.975905418395996
Teacher-sequence CE       : 11.158007621765137
Combined loss             : 11.066956520080566
Student gradient L2       : 6.834960460662842
Student gradients finite  : True
Direct Teacher logits     : False
Teacher model loaded      : False
Test Set status           : SEALED
Progress                  : 4 / 6 code cells
Next                      : discarded optimizer/checkpoint pilot


## Cell 5 — Discarded optimizer step và exact restore

**Công việc**

- thực hiện đúng một FP16 optimizer step;
- xác nhận encoder và decoder đều thay đổi;
- lưu/load checkpoint tạm gồm optimizer, scheduler, scaler và RNG;
- khôi phục chính xác initial Student rồi xóa checkpoint tạm.

**Đầu ra dự kiến:** update và restore `PASSED`, tiến độ `5 / 6`.


In [5]:
STARTED = time.perf_counter()
TEMP_CHECKPOINT = OUTPUT_ROOT / "phase4b_temporary_preflight.pt"


def model_state_cpu(model):
    return {
        key: value.detach().cpu().clone()
        for key, value in model.state_dict().items()
    }


def state_exact(left, right):
    return left.keys() == right.keys() and all(
        torch.equal(left[key], right[key]) for key in left
    )


def cpu_tree(value):
    if torch.is_tensor(value):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {key: cpu_tree(item) for key, item in value.items()}
    if isinstance(value, list):
        return [cpu_tree(item) for item in value]
    if isinstance(value, tuple):
        return tuple(cpu_tree(item) for item in value)
    return copy.deepcopy(value)


def nested_exact(left, right):
    if torch.is_tensor(left) or torch.is_tensor(right):
        return (
            torch.is_tensor(left)
            and torch.is_tensor(right)
            and torch.equal(left.detach().cpu(), right.detach().cpu())
        )
    if isinstance(left, dict) or isinstance(right, dict):
        return (
            isinstance(left, dict)
            and isinstance(right, dict)
            and left.keys() == right.keys()
            and all(
                nested_exact(left[key], right[key]) for key in left
            )
        )
    if isinstance(left, (list, tuple)) or isinstance(
        right, (list, tuple)
    ):
        return (
            isinstance(left, type(right))
            and len(left) == len(right)
            and all(
                nested_exact(a, b) for a, b in zip(left, right)
            )
        )
    return left == right


STUDENT_ENCODER.load_state_dict(INITIAL_ENCODER, strict=True)
STUDENT_DECODER.load_state_dict(INITIAL_DECODER, strict=True)
backbone = list(STUDENT_ENCODER.backbone.parameters())
adapter = [
    *STUDENT_ENCODER.projection.parameters(),
    STUDENT_ENCODER.visual_position_embedding,
    *STUDENT_ENCODER.output_norm.parameters(),
]
optimizer = torch.optim.AdamW(
    [
        {"params": backbone, "lr": 1e-4, "name": "backbone"},
        {"params": adapter, "lr": 3e-4, "name": "adapter"},
        {
            "params": list(STUDENT_DECODER.parameters()),
            "lr": 3e-4,
            "name": "decoder",
        },
    ],
    weight_decay=1e-4,
)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lambda step: min(1.0, (step + 1) / 1771)
)
scaler = torch.amp.GradScaler(
    "cuda", enabled=True, init_scale=1024.0
)
optimizer.zero_grad(set_to_none=True)
set_student_train()
with torch.amp.autocast("cuda", dtype=torch.float16):
    memory = STUDENT_ENCODER(STUDENT_IMAGES)
    human_output = STUDENT_DECODER(
        HUMAN_INPUT, memory, HUMAN_MASK
    )
    pseudo_output = STUDENT_DECODER(
        PSEUDO_INPUT, memory, PSEUDO_MASK
    )
    pilot_loss = (
        0.5 * sequence_ce(human_output, HUMAN_LABELS)
        + 0.5 * sequence_ce(pseudo_output, PSEUDO_LABELS)
    )
assert torch.isfinite(pilot_loss)
scaler.scale(pilot_loss).backward()
scaler.unscale_(optimizer)
gradient_norm = torch.nn.utils.clip_grad_norm_(
    list(STUDENT_ENCODER.parameters())
    + list(STUDENT_DECODER.parameters()),
    1.0,
)
assert torch.isfinite(gradient_norm)
scale_before = scaler.get_scale()
scaler.step(optimizer)
scaler.update()
assert scaler.get_scale() >= scale_before
scheduler.step()

post_encoder = model_state_cpu(STUDENT_ENCODER)
post_decoder = model_state_cpu(STUDENT_DECODER)
assert not state_exact(post_encoder, INITIAL_ENCODER)
assert not state_exact(post_decoder, INITIAL_DECODER)

expected_optimizer = cpu_tree(optimizer.state_dict())
expected_scheduler = cpu_tree(scheduler.state_dict())
expected_scaler = cpu_tree(scaler.state_dict())
payload = {
    "schema_version": "phase4b_expansionnet_preflight_checkpoint_v1",
    "seed": SEED,
    "encoder": post_encoder,
    "decoder": post_decoder,
    "optimizer": expected_optimizer,
    "scheduler": expected_scheduler,
    "scaler": expected_scaler,
    "cpu_rng": torch.get_rng_state(),
    "cuda_rng": torch.cuda.get_rng_state_all(),
    "test_set_status": "sealed",
    "test_set_used": False,
}
torch.save(payload, TEMP_CHECKPOINT)
loaded = torch.load(
    TEMP_CHECKPOINT, map_location="cpu", weights_only=False
)
assert state_exact(loaded["encoder"], post_encoder)
assert state_exact(loaded["decoder"], post_decoder)
assert nested_exact(loaded["optimizer"], expected_optimizer)
assert nested_exact(loaded["scheduler"], expected_scheduler)
assert nested_exact(loaded["scaler"], expected_scaler)
assert torch.equal(loaded["cpu_rng"], torch.get_rng_state())
assert all(
    torch.equal(a, b)
    for a, b in zip(
        loaded["cuda_rng"], torch.cuda.get_rng_state_all()
    )
)
del loaded

STUDENT_ENCODER.load_state_dict(INITIAL_ENCODER, strict=True)
STUDENT_DECODER.load_state_dict(INITIAL_DECODER, strict=True)
assert state_exact(
    model_state_cpu(STUDENT_ENCODER), INITIAL_ENCODER
)
assert state_exact(
    model_state_cpu(STUDENT_DECODER), INITIAL_DECODER
)
TEMP_CHECKPOINT.unlink()
assert not TEMP_CHECKPOINT.exists()

write_json(
    AUDIT_DIR / "phase4b_cell05_discarded_pilot.json",
    {
        "audit_version": "phase4b_expansionnet_cell05_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "optimizer_steps": 1,
        "pilot_loss": float(pilot_loss.detach()),
        "gradient_norm_before_clip": float(gradient_norm),
        "weights_updated": True,
        "encoder_updated": True,
        "decoder_updated": True,
        "checkpoint_restore_exact": True,
        "optimizer_scheduler_scaler_restored": True,
        "rng_restored_exact": True,
        "initial_state_restored_exact": True,
        "temporary_checkpoint_deleted": True,
        "fp16_overflow": False,
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - STARTED,
    },
)
print("=" * 72)
print("PHASE 04B EXPANSIONNET -> S1 CELL 5 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Discarded optimizer steps : 1")
print("Pilot loss                :", float(pilot_loss.detach()))
print("Parameter update observed : True")
print("Encoder / Decoder updated : True / True")
print("FP16 overflow             : False")
print("Checkpoint restore exact  : True")
print("Initial restore exact     : True")
print("Temporary checkpoint      : DELETED")
print("Full training started     : False")
print("Test Set status           : SEALED")
print("Progress                  : 5 / 6 code cells")
print("Next                      : package preflight and authorize 04C")


PHASE 04B EXPANSIONNET -> S1 CELL 5 STATUS
Status                    : PASSED
Discarded optimizer steps : 1
Pilot loss                : 11.008502960205078
Parameter update observed : True
Encoder / Decoder updated : True / True
FP16 overflow             : False
Checkpoint restore exact  : True
Initial restore exact     : True
Temporary checkpoint      : DELETED
Full training started     : False
Test Set status           : SEALED
Progress                  : 5 / 6 code cells
Next                      : package preflight and authorize 04C


## Cell 6 — Production policy, manifest và authorization

**Công việc**

- khóa policy cho ba seed 42/123/2026;
- tạo marker cấp quyền cho Phase 04C training readiness;
- tạo manifest và COMPLETE marker;
- cập nhật tiến độ và độ phù hợp với đề cương.

**Đầu ra dự kiến:** Phase 04B replacement `COMPLETE`, tiến độ `6 / 6`.


In [6]:
STARTED = time.perf_counter()

POLICY_PATH = CONFIG_DIR / (
    "teacher_a_s1_sequence_kd_production_policy.json"
)
policy = {
    "policy_version": "teacher_a_s1_sequence_kd_production_v1",
    "created_at_utc": now_utc(),
    "status": "locked",
    "experiment_id": "P4A_EXPANSIONNET_S1_SEQKD",
    "teacher": "ExpansionNet v2 pseudo-caption corpus",
    "student": "S1 MobileNetV3-Large",
    "method": "sequence_kd",
    "seeds": [42, 123, 2026],
    "supervision": {
        "human_caption_ce_weight": 0.5,
        "teacher_sequence_ce_weight": 0.5,
        "temperature": None,
        "direct_teacher_logits_used": False,
    },
    "training": {
        "images_per_epoch": 113287,
        "physical_batch_size": 32,
        "effective_batch_size": 64,
        "gradient_accumulation_steps": 2,
        "maximum_epochs": 20,
        "optimizer": "AdamW",
        "backbone_lr": 1e-4,
        "adapter_lr": 3e-4,
        "decoder_lr": 3e-4,
        "weight_decay": 1e-4,
        "gradient_clip_norm": 1.0,
        "mixed_precision": "FP16",
        "gradscaler_initial_scale": 1024.0,
        "warmup_epochs": 1,
        "scheduler": "warmup_cosine",
        "minimum_lr_factor": 0.05,
    },
    "selection": {
        "split": "Karpathy Validation only",
        "primary": "CIDEr-D",
        "tie_break_order": ["BLEU-4", "CE", "earlier epoch"],
        "beam_size": 3,
        "length_penalty": 0.7,
        "maximum_decoder_steps": 31,
        "early_stopping": {
            "minimum_epoch": 5,
            "patience": 4,
            "minimum_delta": 0.001,
        },
    },
    "pseudo_caption_parquet": {
        "source_sha256": sha256_file(PSEUDO_PARQUET),
        "records": 113287,
    },
    "feature_kd_allowed": False,
    "test_set_status": "sealed",
}
write_json(POLICY_PATH, policy)

NEXT_AUTH = OUTPUT_ROOT / (
    "PHASE4_TEACHER_A_S1_SEQUENCE_KD_"
    "TRAINING_READINESS_AUTHORIZED.json"
)
write_json(
    NEXT_AUTH,
    {
        "marker_version": (
            "phase4_teacher_a_s1_sequence_kd_"
            "training_readiness_authorized_v1"
        ),
        "created_at_utc": now_utc(),
        "status": "authorized",
        "experiment_id": "P4A_EXPANSIONNET_S1_SEQKD",
        "teacher": "ExpansionNet v2",
        "student": "S1 MobileNetV3-Large",
        "method": "sequence_kd",
        "seeds": [42, 123, 2026],
        "policy": {
            "relative_path": POLICY_PATH.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(POLICY_PATH),
        },
        "next_phase": (
            "Phase 04C Teacher A to S1 Sequence KD "
            "replacement training readiness"
        ),
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

MANIFEST_CSV = MANIFEST_DIR / (
    "phase4b_expansionnet_s1_sequence_kd_manifest.csv"
)
SHA256SUMS = MANIFEST_DIR / "SHA256SUMS"
FINAL_MARKER = OUTPUT_ROOT / (
    "PHASE4_TEACHER_A_S1_SEQUENCE_KD_PREFLIGHT_COMPLETE.json"
)
artifacts = sorted(
    [
        path
        for path in OUTPUT_ROOT.rglob("*")
        if path.is_file()
        and path not in {
            MANIFEST_CSV, SHA256SUMS, FINAL_MARKER
        }
        and not path.name.endswith(".tmp")
    ],
    key=lambda path: path.relative_to(OUTPUT_ROOT).as_posix(),
)
rows = [
    {
        "relative_path": path.relative_to(
            OUTPUT_ROOT
        ).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in artifacts
]
with MANIFEST_CSV.open(
    "w", encoding="utf-8", newline=""
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    writer.writerows(rows)
SHA256SUMS.write_text(
    "".join(
        f'{row["sha256"]}  {row["relative_path"]}\n'
        for row in rows
    ),
    encoding="utf-8",
)
for row in rows:
    artifact = resolve_inside(
        OUTPUT_ROOT, row["relative_path"]
    )
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

write_json(
    FINAL_MARKER,
    {
        "marker_version": (
            "phase4_teacher_a_s1_sequence_kd_"
            "preflight_complete_v1"
        ),
        "created_at_utc": now_utc(),
        "status": "complete",
        "phase": "04B replacement",
        "experiment_id": "P4A_EXPANSIONNET_S1_SEQKD",
        "teacher": "ExpansionNet v2",
        "student": "S1 MobileNetV3-Large",
        "method": "sequence_kd",
        "seed": 42,
        "fixed_images": 64,
        "pseudo_caption_source": "Phase 03E locked corpus",
        "student_checkpoint_load": "passed",
        "combined_loss_backward": "passed",
        "discarded_optimizer_pilot": "passed",
        "atomic_restore": "passed",
        "initial_state_restored": True,
        "direct_logit_kd_allowed": False,
        "old_clip_logit_kd_status": "historical_only",
        "training_started": False,
        "phase04c_authorized": True,
        "next_authorization": {
            "relative_path": NEXT_AUTH.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(NEXT_AUTH),
        },
        "manifest": {
            "csv": {
                "relative_path": MANIFEST_CSV.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(MANIFEST_CSV),
                "rows": len(rows),
            },
            "sha256sums": {
                "relative_path": SHA256SUMS.relative_to(
                    OUTPUT_ROOT
                ).as_posix(),
                "sha256": sha256_file(SHA256SUMS),
            },
        },
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)
print("=" * 72)
print("PHASE 04B EXPANSIONNET -> S1 FINAL STATUS")
print("=" * 72)
print("Status                    : COMPLETE")
print("Teacher                   : A — ExpansionNet v2")
print("Student                   : S1 — MobileNetV3-Large")
print("Method                    : Sequence KD")
print("Fixed supervision         : 64 / 64 PASSED")
print("Student strict load       : 2 / 2 PASSED")
print("Combined loss/backward    : PASSED")
print("Discarded optimizer pilot : PASSED")
print("Atomic/initial restore    : PASSED / PASSED")
print("Direct Logit KD           : PROHIBITED")
print("Old CLIP Logit KD         : HISTORICAL ONLY")
print("Manifest rows             :", len(rows), "VERIFIED")
print("Training started          : False")
print("Test Set status           : SEALED")
print("Phase 04C readiness       : AUTHORIZED")
print("Progress                  : 6 / 6 code cells")
print(
    "Thesis alignment          : KD method/implementation "
    "preflight COMPLETE"
)
print(
    "Next                      : Phase 04C replacement "
    "training readiness"
)
print("Marker                    :", FINAL_MARKER)


PHASE 04B EXPANSIONNET -> S1 FINAL STATUS
Status                    : COMPLETE
Teacher                   : A — ExpansionNet v2
Student                   : S1 — MobileNetV3-Large
Method                    : Sequence KD
Fixed supervision         : 64 / 64 PASSED
Student strict load       : 2 / 2 PASSED
Combined loss/backward    : PASSED
Discarded optimizer pilot : PASSED
Atomic/initial restore    : PASSED / PASSED
Direct Logit KD           : PROHIBITED
Old CLIP Logit KD         : HISTORICAL ONLY
Manifest rows             : 8 VERIFIED
Training started          : False
Test Set status           : SEALED
Phase 04C readiness       : AUTHORIZED
Progress                  : 6 / 6 code cells
Thesis alignment          : KD method/implementation preflight COMPLETE
Next                      : Phase 04C replacement training readiness
Marker                    : /kaggle/working//phase4_teacher_a_s1_sequence_kd_preflight/PHASE4_TEACHER_A_S1_SEQUENCE_KD_PREFLIGHT_COMPLETE.json
